# Tutorial 00 (Duke Heart, Demo): Predict Cardiac Motion From One Labelmap Frame

## Purpose

A self-contained demo of the cardiac motion-prediction pipeline that fetches
every input it needs, so it runs with nothing more than a clone of the
repository:

1. Download the ``Duke-Heart-4DLabelmaps`` gated labelmaps and the pretrained
   ``physicsnemo_mgn_duke_heart_motion`` MeshGraphNet checkpoint with
   ``monai_physio.DownloadData``. The checkpoint directory also carries the
   heart PCA shape model (``pca_model.json``, ``pca_mean_surface.vtp``) that
   network was trained against, so no separate Tutorial 6 run is needed.
2. Contour the reference frame of the held-out case -- the whole heart minus
   its chamber cavities, the structure the shape model describes.
3. Fit that PCA shape model to the surface with
   ``WorkflowFitStatisticalModelToPatient``, giving this patient's PCA
   coefficients and fitted SSM surface.
4. Predict the motion at every gated stage with the pretrained MeshGraphNet,
   carry the reference labelmap through each stage's deformation, and write the
   series as VTP surfaces and one animated USD --
   ``WorkflowInferMovement.process_time_series``.
5. Score each carried labelmap against the frame actually acquired at that
   stage, beside the score of not predicting any motion at all.

The case is ``DUKE_HEART_4D_LABELMAPS.hold_out_case``, which the network never
saw in training, so the score measures generalization rather than recall.
Unlike the ungated scan of the lung demo, this case is gated, so the stages
come from the acquisition and every prediction has a frame to be scored
against.

These labelmaps were made with the ASCardio module of Simpleware; the network
should only be applied to labelmaps segmented the same way (see
``data/Duke-Heart-4DLabelmaps/README.md``).
``tutorial_00_infer_physics_informed_motion_duke_heart_4d_labelmaps.ipynb`` is
the physics-informed counterpart of this demo.

## Data Required

Nothing -- the labelmaps, the pretrained MeshGraphNet weights and the PCA shape
model they were trained against are all downloaded by this notebook.

## Outputs (under ``output/tutorial_00_infer_cardiac_motion_duke_heart_4d_labelmaps/``)

  * ``pm0027_heart_surface.vtp`` -- contoured reference-frame heart
  * ``pm0027_surface.json`` -- this patient's shape parameters
  * ``pm0027_ssm_surface.vtp`` -- the model fitted to the reference frame
  * ``pm0027_surface_s{TTT}_pred.vtp`` -- predicted surface per stage
  * ``pm0027_surface_s{TTT}_warped.mha`` -- reference labelmap carried to that stage
  * ``pm0027_mgn_motion.usd`` -- animated predicted motion
  * ``pm0027_dice.csv`` -- Dice per stage, predicted and no-motion

In [ ]:
# Imports
import csv
import json
import logging
from typing import cast

import itk
import numpy as np
import pyvista as pv
from parameters_duke_heart_4d_labelmaps import DUKE_HEART_4D_LABELMAPS

from monai_physio import (
    DownloadData,
    EvaluateMovementDukeHeart,
    WorkflowFitStatisticalModelToPatient,
    WorkflowInferMovement,
    WorkflowInferPhysicsNeMo,
)

In [ ]:
# Data directory specification
test_mode = False

case_id = DUKE_HEART_4D_LABELMAPS.hold_out_case

data_dir = DUKE_HEART_4D_LABELMAPS.hold_out_directory(test_mode)
weights_dir = DUKE_HEART_4D_LABELMAPS.weights_directory(test_mode)
output_dir = (
    DUKE_HEART_4D_LABELMAPS.output_directory(test_mode)
    / "tutorial_00_infer_cardiac_motion_duke_heart_4d_labelmaps"
)

# Weights Tutorial 9 trains; this demo downloads a pretrained copy instead.
# The PCA model and mean surface it was trained against ship in the same
# checkpoint directory.
model_dir = DUKE_HEART_4D_LABELMAPS.mgn_weights_directory(test_mode)
epoch = None  # None uses the final weights
pca_model_file = model_dir / "pca_model.json"
pca_mean_file = model_dir / "pca_mean_surface.vtp"

number_of_pca_components = DUKE_HEART_4D_LABELMAPS.pca_components(test_mode)

# Gaussian sigma, in mm, that spreads the predicted surface displacements
# into the continuous field the labelmap is resampled through.
smoothing_sigma_mm = 10.0

log_level = logging.INFO
logging.basicConfig(level=log_level)
logger = logging.getLogger("tutorial_00_infer_cardiac_motion_duke_heart_4d_labelmaps")

output_dir.mkdir(parents=True, exist_ok=True)

## Step 1: Download the labelmaps and the pretrained MeshGraphNet weights

In [ ]:
DownloadData.DownloadDukeHeart4DLabelmapsData(data_dir)
DownloadData.DownloadPhysicsNeMoMGNDukeHeartMotionData(weights_dir)

pca_mean_surface = cast(pv.DataSet, pv.read(str(pca_mean_file)))
with pca_model_file.open(encoding="utf-8") as f:
    pca_model = json.load(f)

## Step 2: Read the gated frames and contour the reference frame

Each frame's ``g{PPP}`` gate tag gives its cardiac stage. The frame tagged
``_ref`` is the one the network's displacements start from, since that is the
frame its training displacements were measured from.

In [ ]:
frame_files = sorted((data_dir / case_id).glob("*_labelmap.nii.gz"))
reference_file = next(
    path for path in frame_files if path.name.endswith("_ref_labelmap.nii.gz")
)
cohort = EvaluateMovementDukeHeart(log_level=log_level)
stages = [cohort.stage_from_filename(path) for path in frame_files]
reference_labelmap = itk.imread(str(reference_file))
logger.info("Case %s: %d gated frames", case_id, len(frame_files))

heart_surface = DUKE_HEART_4D_LABELMAPS.whole_heart_surface(
    reference_file, test_mode, log_level
)
heart_surface.save(str(output_dir / f"{case_id}_heart_surface.vtp"))

## Step 3: Fit the heart shape model to the reference frame

The PCA coefficients are what the network is conditioned on; the fitted
surface is what its displacements are added to. This data carries no intensity
image, so the workflow rasterizes its own reference grid from the surface.

In [ ]:
fit_workflow = WorkflowFitStatisticalModelToPatient(
    template_model=pca_mean_surface,
    patient_models=[heart_surface],
    patient_image=None,
    patient_labelmap=reference_labelmap,
    labelmap_interior_object_ids=DUKE_HEART_4D_LABELMAPS.interior_object_ids,
    log_level=log_level,
)
fit_workflow.set_use_pca_registration(
    use_pca_registration=True,
    pca_model=pca_model,
    number_of_pca_components=number_of_pca_components,
    use_surface=False,
)
fit_workflow.set_icp_transform_type(DUKE_HEART_4D_LABELMAPS.icp_transform_type)
fit_workflow.set_mask_dilation_mm(DUKE_HEART_4D_LABELMAPS.mask_dilation_mm)
fit_workflow.set_distancemap_squared_max(
    DUKE_HEART_4D_LABELMAPS.distancemap_squared_max
)
fit_result = fit_workflow.process()

pca_coefficients = fit_workflow.pca_coefficients
assert pca_coefficients is not None
# WorkflowInferMovement.process_time_series names every predicted surface
# after this file's stem, so naming it "{case_id}_surface" is what gives the
# predicted surfaces their "{case_id}_surface_s{TTT}_pred.vtp" names below.
pca_coefficients_file = output_dir / f"{case_id}_surface.json"
with pca_coefficients_file.open(mode="w", encoding="utf-8") as f:
    json.dump(pca_coefficients.tolist(), f)

# The heart PCA model is built from surfaces only, so the model *is* a
# surface here: only the .vtp is written.
fitted_reference_mesh_file = output_dir / f"{case_id}_ssm_surface.vtp"
fit_result["fitted_reference_mesh"].save(str(fitted_reference_mesh_file))
logger.info("Fitted the heart model to %s", reference_file.name)

## Step 4: Predict cardiac motion with the pretrained MeshGraphNet

Predict every gated stage, carry the reference labelmap through each stage's
deformation, and write the animated USD. Nearest-neighbor interpolation keeps
the label values discrete. The shape model is one structure, so the USD
surface is kept whole rather than split by connectivity.

In [ ]:
infer_workflow = WorkflowInferPhysicsNeMo(
    model_directory=model_dir, epoch=epoch, log_level=log_level
)
infer_result = WorkflowInferMovement(
    infer_workflow, log_level=log_level
).process_time_series(
    shape_parameters=pca_coefficients_file,
    stages=stages,
    output_directory=output_dir,
    fitted_reference_mesh=fitted_reference_mesh_file,
    reference_image=reference_labelmap,
    warp_interpolation="nearest",
    warp_background_value=0.0,
    smoothing_sigma_mm=smoothing_sigma_mm,
    usd_project_name=f"{case_id}_mgn_motion",
    anatomy_type="heart",
    separate_by_connectivity=False,
)
logger.info("USD: %s", infer_result["usd_file"])

## Step 5: Score the prediction against the acquired frames

Every frame of a case shares one image grid, so each carried labelmap is
compared with the frame acquired at its stage voxel by voxel: Dice per label,
averaged over the labels in that frame. The *no-motion* column scores the
reference frame itself against every frame, the result of predicting no motion
at all; the prediction earns its keep where it beats that column, which is
furthest from the reference frame in the cycle.

In [ ]:
def mean_label_dice(predicted: np.ndarray, acquired: np.ndarray) -> float:
    """Return the Dice of each label in ``acquired``, averaged over labels."""
    scores = []
    for label in np.unique(acquired):
        if label == 0:
            continue
        predicted_mask = predicted == label
        acquired_mask = acquired == label
        overlap = np.logical_and(predicted_mask, acquired_mask).sum()
        scores.append(2.0 * overlap / (predicted_mask.sum() + acquired_mask.sum()))
    return float(np.mean(scores))


reference_array = itk.array_from_image(reference_labelmap)
rows = []
for stage, frame_file, warped_file in zip(
    stages, frame_files, infer_result["warped_images"]
):
    acquired = itk.array_from_image(itk.imread(str(frame_file)))
    predicted = itk.array_from_image(itk.imread(str(warped_file)))
    rows.append(
        {
            "stage": stage,
            "dice_predicted": mean_label_dice(predicted, acquired),
            "dice_no_motion": mean_label_dice(reference_array, acquired),
        }
    )

dice_file = output_dir / f"{case_id}_dice.csv"
with dice_file.open(mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

print(f"{'stage':>6} {'predicted':>10} {'no motion':>10}")
for row in rows:
    print(
        f"{row['stage']:6.2f} {row['dice_predicted']:10.3f} "
        f"{row['dice_no_motion']:10.3f}"
    )

## Step 6: Look at the fitted and predicted surfaces

The predicted surface is shown at the stage furthest from the reference
frame -- the one the no-motion column scores lowest.

In [ ]:
furthest = int(np.argmin([row["dice_no_motion"] for row in rows]))

pv.set_jupyter_backend("static")

plotter = pv.Plotter(shape=(1, 2), window_size=[1000, 500])
plotter.subplot(0, 0)
plotter.add_mesh(
    cast(pv.DataSet, pv.read(str(fitted_reference_mesh_file))),
    color="steelblue",
)
plotter.add_text("Fitted reference surface", font_size=10)
plotter.camera_position = "iso"

plotter.subplot(0, 1)
plotter.add_mesh(
    cast(pv.DataSet, pv.read(str(infer_result["predicted_surfaces"][furthest]))),
    color="limegreen",
)
plotter.add_text(f"Predicted surface, stage {stages[furthest]}", font_size=10)
plotter.camera_position = "iso"

plotter.show()